In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,HfArgumentParser,TrainingArguments,pipeline, logging
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
import os,torch
import bitsandbytes as bnb
from datasets import load_dataset
from trl import SFTTrainer
from datasets import Dataset
import pyarrow as pa
import pyarrow.dataset as ds
import pandas as pd
import re
import wandb
from utils import * 

2024-10-17 11:04:19.636115: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-17 11:04:19.636177: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-17 11:04:19.637389: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-17 11:04:19.647554: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-17 11:04:22.021150: W tensorflow/compiler/tf2

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
print("CUDA devices: ",torch.cuda.device_count())

# Get the number of GPUs
num_gpus = torch.cuda.device_count()

# Iterate through each GPU and print its details
for i in range(num_gpus):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  CUDA capability: {torch.cuda.get_device_capability(i)}")
    print(f"  Total memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
    print(f"  Current memory allocated: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
    print(f"  Current memory cached: {torch.cuda.memory_reserved(i) / 1e9:.2f} GB\n")


In [3]:
MODEL_PATH = "./aya-23-8b"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    #quantization_config=bnb_config,
    trust_remote_code=True,
    #device_map='auto',   
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
device = torch.cuda.current_device()
model = model.to(device)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

In [6]:
def build_corpus(data_dir):
    
    # List to store the extracted columns
    columns_list = []

    # Loop through each file in the folder
    for file in os.listdir(data_dir):
        if file.endswith('.csv'):
            # Read the CSV file
            df = pd.read_csv(os.path.join(data_dir, file))

            # Extract the column and append it to the list
            columns_list = columns_list + df['apc_Arabic'].tolist()

    return columns_list

In [59]:
len(build_corpus('./datasets/corpus_data'))

71279

In [7]:
def get_few_shot(few_shot_dir):
    ar_list = []
    en_list=[]
    for file in os.listdir(few_shot_dir):
        if file.endswith('.csv'):
            df = pd.read_csv(os.path.join(few_shot_dir, file))
            ar_list = ar_list + df['apc_Arabic'].tolist()
            en_list = en_list + df['English'].tolist()

    return ar_list, en_list

In [18]:
def get_response(question, tokenizer,model,examples_ar, examples_en):
    top_5_indices = get_top_indices(question, examples_ar)
    top_5_examples_ar = np.array(examples_ar)[top_5_indices]
    print(top_5_examples_ar)
    top_5_examples_en = np.array(examples_en)[top_5_indices]
    print(top_5_examples_en)
    prompt = get_message_format(question, top_5_examples_ar, top_5_examples_en)
    input_ids = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt", padding= True)
    inputs = input_ids.to(device)

    #prompt_padded_len = len(input_ids[0])
    
    gen_tokens = model.generate(
        inputs,
        max_new_tokens=400, 
        do_sample=True, 
        temperature=0.4,
    )
    
    response = tokenizer.decode(
        gen_tokens[0], skip_special_tokens=True, clean_up_tokenization_spaces=True
    )
    
    response = response.split("<|CHATBOT_TOKEN|>English:")[-1].strip()
    #response =  response.split("rather than")[0].strip()
    print(response)
    return response

generations = []
for input in inputs: 
    generations.append(get_response(input,tokenizer, model, examples_ar, examples_en))

NameError: name 'inputs' is not defined

In [51]:
import numpy as np

def tokenize_arabic(text):
    return re.findall(r'\S+', text)


def extract_ngrams(text, n):
    # Tokenize the text into words
    words = tokenize_arabic(text)
    # Generate n-grams
    n_grams = [tuple(words[i:i+n]) for i in range(len(words)-n+1)] 
    return set(n_grams)

def compare_ngrams(sentence1, sentence2,n):
    ngrams1 = extract_ngrams(sentence1,n)
    ngrams2 = extract_ngrams(sentence2,n)
    matching_ngrams = ngrams1.intersection(ngrams2)
    return matching_ngrams

def get_matching_n_indices(question, examples_ar, n):
    matching_indices = []
    ngrams = set()  # Using a set for faster lookups

    for index, example in enumerate(examples_ar):
        matching_ngrams = compare_ngrams(question, example, n)
        if matching_ngrams:
            new_match = False
            for ngram in matching_ngrams:
                joined_ngram = " ".join(ngram)
                if joined_ngram not in ngrams:
                    print(joined_ngram)
                    ngrams.add(joined_ngram)
                    new_match = True
            
            if new_match and len(matching_indices) < 5:
                matching_indices.append(index)

    return matching_indices

def build_corpus_frequency(corpus):
    all_words = []
    for sentence in corpus:
        all_words.extend(tokenize_arabic(sentence))
    return Counter(all_words)


def get_rare_words(words, corpus_frequency, rarity_threshold):
    return [word for word in words if corpus_frequency[word] <= rarity_threshold]

def find_matching_rare_words_indices(question, examples_ar, corpus_frequency, rarity_threshold):
    matching_indices = []
    unique_words=set()
    rare_words_question = set(get_rare_words(set(tokenize_arabic(question)), corpus_frequency, rarity_threshold))
    
    for index, example in enumerate(examples_ar):
        words_example = set(tokenize_arabic(example))
        rare_words_example = set(get_rare_words(words_example, corpus_frequency, rarity_threshold))
        
        matching_rare_words = rare_words_question.intersection(rare_words_example)
        
        if matching_rare_words:
            for word in matching_rare_words:
                if word not in unique_words:
                    print(f"{word} (frequency in corpus: {corpus_frequency[word]})")
                    unique_words.add(word)
                    matching_indices.append(index)
    
    return unique_words,matching_indices


def get_examples_translations(indices, examples_ar,examples_en):
    examples = np.array(examples_ar)[indices]
    translations = np.array(examples_en)[indices]
    
    return examples,translations

In [29]:
def generate_prompt(question, examples, translations, rare_words):
    # The initial part of the prompt
    prompt_start = '''You are a skilled translator with expertise in Lebanese colloquial language. It is very important to focus on correctly translating the following words {rare_words}

Examples:
'''
    prompt_start = prompt_start.replace('rare_words', ', '.join(rare_words))
    # Generate the examples section
    examples_section = ''
    for ex, trans in zip(examples, translations):
        examples_section += f'\n\nInput:{ex}\nOutput:{trans}'

    # Combine all parts of the prompt
    full_prompt = prompt_start + examples_section + f'\n\nInput:{{Question}}\nOutput:'

    # Replace the {Question} placeholder with the actual question
    full_prompt = full_prompt.replace('{Question}', question)
    
    message= [{"role": "user", "content": full_prompt}]

    return message

In [17]:
def get_response(question, tokenizer,model,device,examples_ar, examples_en,corpus_frequency,rarity_threshold,n):

    ngrams_indices= get_matching_n_indices(question, examples_ar, n)
    rare_words,rare_indices= find_matching_rare_words_indices(question, examples_ar, corpus_frequency, rarity_threshold)
    examples_ngrams, translations_ngrams= get_examples_translations(ngrams_indices, examples_ar,examples_en)
    examples_rare,translations_rare= get_examples_translations(rare_indices, examples_ar,examples_en)
    examples= np.concatenate((examples_ngrams,examples_rare))
    translations= np.concatenate((translations_ngrams,translations_rare))
    prompt = generate_prompt(question, examples,translations,rare_words)
    print(prompt)
    input_ids = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt", padding= True)
    inputs = input_ids.to(device)

    #prompt_padded_len = len(input_ids[0])
    
    gen_tokens = model.generate(
        inputs,
        max_new_tokens=400, 
        do_sample=True, 
        temperature=0.4,
    )
    
    response = tokenizer.decode(
        gen_tokens[0], skip_special_tokens=True, clean_up_tokenization_spaces=True
    )
    
    response = response.split("<|CHATBOT_TOKEN|>")[-1].strip()
    print('Response\n', response)
    #response =  response.split("rather than")[0].strip()
    return response


In [42]:
data_dir= './datasets/corpus_data'
corpus_data= build_corpus(data_dir)
few_shot_dir = './datasets/Few-shot-data'
test_data = pd.read_csv('./datasets/test_data.csv')


corpus_frequency = build_corpus_frequency(corpus_data)
examples_ar, examples_en = get_few_shot(few_shot_dir)
inputs = test_data['apc_Arabic'].tolist()

In [53]:
rarity_threshold = 5
n=2
question=  "عادة بعمل فنجان قهوة وبشربو على البلكون، بحب أستمع كتير على فيروز. أنا وعم أشرب القهوة برد على الإيميلات. بس خلص فنجان القهوة بروح عل مطبخ وبعمل بيض مقلي."

In [54]:
#initial prompt
get_response(question,tokenizer,model,device,examples_ar, examples_en,corpus_frequency,rarity_threshold,n)

كتير على
أنا وعم
فنجان قهوة
مطبخ (frequency in corpus: 4)
[{'role': 'user', 'content': 'You are a skilled translator with expertise in Lebanese colloquial language. It is very important to focus on correctly translating the following words {مطبخ}\n\nExamples:\n\n\nInput:ستّا هون ضحكت كتير على شبه البنت لإما حتى بهالشغلة وقالت “طب الجرة عتما بتطلع البنت لإما”.\nOutput:The grandmother here laughed a lot at how much the little girl resembled her mother even in this detail, and she said “Drop the pitcher on its mouth, the girl would resemble her mother.”\n\nInput:وبما إنو أنا وعم حضر هالحلقة، شكلو الولد اللي فيي عم يتغلب على عقلي الراشد، قررت إحكي عن عيد الميلاد بنسختو اللبنانة، أو كيف منعيّد الميلاد بلبنان.\nOutput:And since my inner child seems to be taking over my adult mind while I am preparing this episode, I have decided to talk about the Lebanese version of Christmas, or how we celebrate Christmas in Lebanon.\n\nInput:بكير قمت اليوم تا لحق كل شغلي. عملت فنجان قهوة وطلعت عالشغل قبل م

'I usually make a cup of coffee and drink it on the balcony, I like to listen to a lot of Fairouz. I and my father drink coffee while checking our emails. But when I finish the cup of coffee, I go to the kitchen and cook fried eggs.'

In [ ]:
def get_message_format(text):
    prompt = "You are a skilled translator with expertise in Lebanese colloquial language, its grammar and its vocabulary. Translate this sentence from Lebanese Arabic to English.\nInput:{Question}"
    #prompt = "Translate this sentence from Lebanese Arabic to English.\nInput:{Question}"
    #prompt = 'You are a skilled translator with expertise in Lebanese colloquial language. To help you with your translation, you are given grammatical rules for the present, past, and future tenses in Lebanese. Make use these rules to avoid mistakes in translating verb tenses.\n\nPresent Tense Rules:\nGeneral Actions (Present Simple): Grammar Point: Present tense conjugation only. Usage: Use this form to express things you do in general. Example: Levantine: انا بشرب قهوة كل يوم الصبح. English: I drink coffee every morning. \n Ongoing Actions (Present Continuous) Grammar Point: Present tense conjugation + عم Usage: Use this to express something happening at this moment. Example: Levantine: انا عم بشرب قهوة هلق English: I am drinking coffee right now. \n \n Past Tense Rules: \n Completed Actions (Past Simple)\nGrammar Point: Past tense conjugation.\nUsage: Use this to express that an action is fully completed. \nExample: Levantine: اتغديت عند بيت صحابي مبارح English: I ate lunch at my boyfriend’s house yesterday. \n Ongoing Past Actions \n Grammar Point: Present tense conjugation + كان + عم \nUsage: Use this to express an action that happened continuously in the past. \nExample: Levantine: الاسبوع الماضي كنت عم بتغدى عند بيت صحابي كل يوم لما ماما كانت برحلة English: Last week, I was eating lunch at my boyfriend’s house every day when my mother was on a trip. \nInterrupting Past Actions \nGrammar Point: Present tense conjugation + لما + كان + عم \nUsage: Use this when one action was happening and another interrupted it.\nExample: Levantine: مبارح لما كنت عم بتغدى عند بيت صحابي امي دقتلي English: Yesterday, when I was eating lunch at my boyfriend’s house, my mother called to ask where I was. \nHabitual Past Actions (Used to) \nGrammar Point: Present tense conjugation + كان \nUsage: Use this to express habitual actions in the past. \nExample: Levantine: كنت اتغدى عند بيت صحابي كل يوم بس امي اكتشفت هذا الشي فا بطلت English: I used to eat lunch at my boyfriend’s house, but my mother found out, so I stopped. \n\n Future Tense Rules: \nFuture Actions (Will) \nGrammar Point: Present tense conjugation + رح\nUsage: Use this to express future actions.\nExample: Levantine: بكرا رح اشوفك عالساعة الخامسة English: I will see you at 5 pm tomorrow. \nNegating Future Actions (Will Not) \nGrammar Point: Present tense conjugation + ما + رح \nUsage: Use this to express actions that will not happen in the future. \nExample: Levantine: بكرا ما رح اشوفك عالساعة الخامسة English: I will not see you at 5 pm tomorrow. \nConditional Future Actions \nGrammar Point: رح + Present tense conjugation + بس\nUsage: Use this to express something that will happen when another action is completed. \nExample:Levantine: بكرا بس اخلص من الشغل رح اشوفكEnglish: Tomorrow, when I get out of work, I will see you.\n\nTask:Translate the following sentence from Lebanese to English. Use the grammatical rules provided. Only provide the translation with no additional explanation : \nInput:{Question}'
    message = prompt.format(Question=text)
    message= [{"role": "user", "content": message}]
    return message

In [ ]:
'''
    few_shot_prompt= 'You are a skilled translator with expertise in Lebanese colloquial language. Use the grammatical Lebanese rules and the rich Lebanese vocabulary to translate an inuput sentence to English. Remember to focus on conveying the meaning of idiomatic expressions popular in the Lebanese culture and to apply the Lebanese Grammatical rules.\
                     \n\nExamples:\n\nInput:الرقص الشرقي فن جميل كتير. كل ما بروح على عرس بلبنان بنبسط لما بشوف الناس عم ترقص دبكة.\nOutput:Belly dancing is a very beautiful art. Every time I go to a wedding in Lebanon, I enjoy seeing people dancing dabke\
                     \n\nInput:لمّا كنت صغير كنت انزل على الدكان لحالي، وكنت اشتري سكاكر وبوظة من عم سليم يلي كان دايماً يعطيني حبة زيادة.\nOutput:When I was young, I used to go to the shop by myself, and I would buy candies and ice cream from Uncle Salim who always gave me an extra piece\
                     \n\nInput: لو ساقو بشكل منيح، ما كانو عملو حادث على طريق جونية. هلأ السيارة مكسرة وهني بالمستشفى عم ياخدو علاج.\nOutput:If they had driven properly, they would not have had an accident on the Jounieh road. Now the car is wrecked and they are in the hospital receiving treatment.\
                     \n\nInput:بيارتة معروفين بحبن للحياة وكرمن. بيحبو يطلعو ويسهرو ويوكلو منيح. إذا زرت بيروت مرة، لازم تجرب المطاعم والمقاهي تبعا. رح تنبسط كتير.\nOutput:Beirutis are known for their love of life and generosity. They like to go out, stay up late, and eat well. If you visit Beirut once, you must try its restaurants and cafes. You will enjoy it a lot\
                     \n\nInput: بيمشي جاري كل صباح على كورنيش بيروت. بيقول انو المشي جنب البحر بيعطيه طاقة لليوم كلو. بيحب يشوف الناس وهني عم يمارسوا الرياضة.\nOutput:My neighbor walks every morning on Beirut Corniche. He says walking by the sea gives him energy for the whole day. He likes to see people exercising\
                     \n\nInput:ما بتعرف أديش بحب الفتوش! بس ما بحب غيرو من السلطات. إمي بتعمل أحلى فتوش بالدني وما في حدا بيعمل متلا.\nOutput:You do not know how much I love fattoush! But I do not like any other salads. My mom makes the best fattoush in the world and no one makes it like her.\
                     \n\nInput:{Question}'
   '''

In [ ]:
def get_message_format(text, translation):
    prompt = "\nExamples:\n<Lebanese>ما اتفقنا تحكيلي حكاية ابريق الزيت!<English>We agreed that you tell me the tale of the oil jug!<Wrong Translation>We agreed that I tell me the tale of the oil jug!<error> I <instead of> we\n<Lebanese><English><Wrong Translation><error><Lebanese>بتمنا تكونو حبيتو حلقتنا لليوم، ونحنا كالعادة منحب نسمع اقترحاتكن وتعليقاتكن على حلقاتنا.<English>I hope you enjoyed our episode for today, and like always we love to hear your suggestions and comments on our episodes.<Wrong Translation>We hope you enjoyed our episode for today, and like always we love to hear your suggestions and comments on our episodes.<error>We <instead of> I. <Lebanese>{Question}<English>{translation}<Wrong Translation>"
    system_prompt= "You are a translation assistant. Your job is to deliberately introduce an error in the pronoun translation of a sentence."
    message = prompt.format(Question=text, translation=translation)
    message= [{"system":system_prompt ,"role": "user", "content": message}]
    return message

In [ ]:
text = "عادة بعمل فنجان قهوة وبشربو على البلكون، بحب أستمع كتير على فيروز. أنا وعم أشرب القهوة برد على الإيميلات.و بس خلص فنجان القهوة رح روح عل مطبخ وبعمل بيض مقلي."

In [ ]:
text = 'ما كان في سيارة عالطريق'

In [ ]:
text = 'كتير بمبسط لما بتعرف على ناس بيحكوا عربي وهني مش عرب.'

In [ ]:
text = 'رح نزعل عليك'

In [ ]:
#initial prompt
get_response(text,tokenizer=tokenizer, model=model, device=device)

In [35]:
#The range of examples, we can extend it to more than LW, mainly to all examples we have
few_shot_data = pd.read_csv('./datasets/language_wave_story_data_sentences.csv')
test_data = pd.read_csv('./datasets/test_data.csv')

examples_ar = few_shot_data['apc_Arabic'].tolist()
examples_en = few_shot_data['English'].tolist()
inputs = test_data['apc_Arabic'].tolist()


#The corpus from which we choose the rarity of a word (extend it to all corpus that we have/all datasets)
corpus_frequency = build_corpus_frequency(examples_ar)
#How many times the word occurs in a certain corpus 
rarity_threshold = 2
#matched ngrams, I am here using bigrams
n = 2

#question=  "كل واحد بيزعل بطريقة معينة. ممكن تضلك زعلان سنين، معلش. خود وقتك!"

generations= []

for input in inputs: 
    generations.append(get_response(input,tokenizer,model,device,examples_ar, examples_en,corpus_frequency,rarity_threshold,n))

عايش (frequency in corpus: 2)
تفوت (frequency in corpus: 1)
عشان (frequency in corpus: 1)
Response
 I graduated from school and now I have to go to the university, of course I am excited and happy because you are now big and independent and you can live wherever you want. You wake up when you want, you sleep when you want, you eat what you want, you dress up when you want, and you do whatever you want!
Response
 Wait a second! Do what you want? Your parents want you to become an engineer, a doctor, or even a lawyer. How do you feel?
بدّو (frequency in corpus: 1)
مستقبلو (frequency in corpus: 1)
معقول (frequency in corpus: 2)
مصلحة (frequency in corpus: 1)
Response
 Of course, parents want their children to have a stable job, but in many cultures, parents force their children to study certain majors. These majors are usually engineering, medicine, or law. Is it possible for the student to choose the major they want? This is also their future.
فهني (frequency in corpus: 1)
نلاقي (frequen

In [36]:
translations = pd.DataFrame(generations, columns=["translations"])
translations.to_csv('few-shot-with-rare-words-n-grams.csv') 